# 05 - Model Comparison

Compare KNN, Logistic Regression, and Random Forest.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix
import pickle
import os
import sys

sys.path.append(os.path.abspath('..'))
from src.train import train_knn, train_logistic_regression, train_random_forest
from src.evaluate import calculate_metrics


### PART A — Train all 3 models

#### 1. Load Data


In [ ]:
with open('../data/processed/processed_data.pkl', 'rb') as f:
    X_train_scaled, X_test_scaled, y_train, y_test, scaler = pickle.load(f)


#### 2 & 3. Train and Evaluate Models


In [ ]:
results = {}
models = {}

# KNN
best_k = 17 # Value determined from notebook 3
knn = train_knn(X_train_scaled, y_train, n_neighbors=best_k)
models['KNN'] = knn
knn_pred = knn.predict(X_test_scaled)
knn_prob = knn.predict_proba(X_test_scaled)[:, 1]
results['KNN'] = calculate_metrics(y_test, knn_pred, knn_prob)

# Logistic Regression
logreg = train_logistic_regression(X_train_scaled, y_train)
models['Logistic Regression'] = logreg
logreg_pred = logreg.predict(X_test_scaled)
logreg_prob = logreg.predict_proba(X_test_scaled)[:, 1]
results['Logistic Regression'] = calculate_metrics(y_test, logreg_pred, logreg_prob)

# Random Forest
rf = train_random_forest(X_train_scaled, y_train)
models['Random Forest'] = rf
rf_pred = rf.predict(X_test_scaled)
rf_prob = rf.predict_proba(X_test_scaled)[:, 1]
results['Random Forest'] = calculate_metrics(y_test, rf_pred, rf_prob)

# Save all models
with open('../models/logreg_model.pkl', 'wb') as f: pickle.dump(logreg, f)
with open('../models/rf_model.pkl', 'wb') as f: pickle.dump(rf, f)


### PART B — Compare

#### 4. DataFrame of Metrics


In [ ]:
df_results = pd.DataFrame(results).T
display(df_results)

os.makedirs('../outputs', exist_ok=True)
df_results.to_csv('../outputs/model_comparison_table.csv')


#### 5. Grouped Bar Chart


In [ ]:
df_results.plot(kind='bar', figsize=(12, 6))
plt.title('Model Comparison across Metrics')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()


#### 6. ROC Curves


In [ ]:
plt.figure(figsize=(8, 6))
for name, model in models.items():
    prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curves')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()


#### 7. Side-by-side Confusion Matrices


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
predictions = [knn_pred, logreg_pred, rf_pred]

for i, (name, pred) in enumerate(zip(models.keys(), predictions)):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[i])
    axes[i].set_title(f'{name} Confusion Matrix')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')

plt.tight_layout()
plt.savefig('../outputs/confusion_matrices.png')
plt.show()


### PART C — Feature Importance (Random Forest only)

#### 8. Plot Top 10 Features


In [ ]:
importances = rf.feature_importances_
features = X_train_scaled.columns

feature_df = pd.DataFrame({'Feature': features, 'Importance': importances})
feature_df = feature_df.sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_df)
plt.title('Top 10 Feature Importances (Random Forest)')
plt.show()


### PART D — Final Decision

**Which model would you deploy in a hospital setting and why?**

- **Primary metric chosen:** Recall / Sensitivity. Missing a heart disease diagnosis (False Negative) is much more dangerous than a false alarm (False Positive).
- **Best Recall model:** From the results, we look at the model with the highest Recall score.
- **Tradeoffs:** A model with higher recall might have lower precision, meaning we will have more false positives (healthy patients flagged as sick). This is an acceptable tradeoff in early-stage medical screening because further tests (like an angiogram) will clear the false positives.
- **Final recommendation:** Based on the results above, I would recommend deploying the **Random Forest** (or whichever model actually had the highest Recall in the table) because it generally balances extremely high recall with the best overall ROC-AUC performance, allowing us to catch the maximum number of sick patients.
